## Mount google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load the model

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch


base_model = "microsoft/deberta-v3-base"

model_path = "/content/drive/MyDrive/deberta_model"

print(f"Attempting to load model from path: {model_path}")

try:
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    # Check for GPU and move model to GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Using device: {device}")

    print("Model and tokenizer loaded successfully!")
except Exception as e:
    print(f"An error occurred while loading the model or tokenizer: {e}")

Attempting to load model from path: /content/drive/MyDrive/deberta_model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
 

Using device: cuda
Model and tokenizer loaded successfully!


In [ ]:
import pandas as pd

# Load data from the Google Sheet URL
sheet_url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vTFY2TDwlxcUMfO6JMkZPRBT_yvan8aLD1o0vzTMR_TX1uoGM6T7xoZF01yMV-XK7s1JxShp4MLwJ_J/pub?output=csv"
try:
    df = pd.read_csv(sheet_url)
    print("Data loaded successfully!")
    display(df.head())
except Exception as e:
    print(f"An error occurred while loading the data: {e}")


Data loaded successfully!


,review_id,sentence_num,sentiment,sentence
0,1548,2,NaN,I just going to enjoy while it lasts.
1,1,2,NaN,This exerciser certainly checked all the boxes...
2,1,3,NaN,The seat position is extremely comfortable and...
3,1,4,NaN,Provides a good range of resistance from super...
4,1,5,NaN,And the resistance bands and the multiple posi...


In [ ]:
import torch
from concurrent.futures import ThreadPoolExecutor
import threading
import math

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- CONFIG ---
BATCH_SIZE = 32         # Try 16, 32, or 64 depending on GPU memory
NUM_THREADS = 4         # 4–6 threads usually best
LOG_INTERVAL = 1000     # Print progress every N sentences

# Thread-safe progress counter
progress_lock = threading.Lock()
completed = 0


Using device: cuda


In [ ]:
def classify_batch(batch_texts):
    """Run inference on a batch of sentences."""
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().tolist()

    return preds


def process_chunk(chunk):
    """Process one chunk of data in multiple batches, thread-safe."""
    global completed
    results = []
    for i in range(0, len(chunk), BATCH_SIZE):
        batch = chunk[i:i + BATCH_SIZE]
        preds = classify_batch(batch)
        results.extend(preds)

        # Progress logging
        with progress_lock:
            completed += len(batch)
            if completed % LOG_INTERVAL < BATCH_SIZE or completed == len(sentences):
                print(f"Processed {completed}/{len(sentences)} sentences...", flush=True)
    return results


In [ ]:
if 'sentence' in df.columns:
    sentences = df['sentence'].dropna().tolist()
    total = len(sentences)
    print(f"Starting classification for {total} sentences...")

    # Split sentences into thread chunks
    chunk_size = math.ceil(total / NUM_THREADS)
    chunks = [sentences[i:i + chunk_size] for i in range(0, total, chunk_size)]

    classified_sentiments = []

    # Threaded execution
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        futures = [executor.submit(process_chunk, chunk) for chunk in chunks]

        for future in futures:
            try:
                classified_sentiments.extend(future.result())
            except Exception as e:
                print(f"Error during batch inference: {e}")

    # Attach predictions to DataFrame
    df = df.copy()
    df.loc[df['sentence'].notna(), 'sentiment'] = classified_sentiments
    print("✅ Sentiment classification complete using batched multithreading.")
    display(df.head())

else:
    print("❌ 'sentence' column not found in the DataFrame.")


Starting classification for 529111 sentences...
Processed 1024/529111 sentences...
Processed 2016/529111 sentences...
Processed 3008/529111 sentences...
Processed 4000/529111 sentences...
Processed 5024/529111 sentences...
Processed 6016/529111 sentences...
Processed 7008/529111 sentences...
Processed 8000/529111 sentences...
Processed 9024/529111 sentences...
Processed 10016/529111 sentences...
Processed 11008/529111 sentences...
Processed 12000/529111 sentences...
Processed 13024/529111 sentences...
Processed 14016/529111 sentences...
Processed 15008/529111 sentences...
Processed 16000/529111 sentences...
Processed 17024/529111 sentences...
Processed 18016/529111 sentences...
Processed 19008/529111 sentences...
Processed 20000/529111 sentences...
Processed 21024/529111 sentences...
Processed 22016/529111 sentences...
Processed 23008/529111 sentences...
Processed 24000/529111 sentences...
Processed 25024/529111 sentences...
Processed 26016/529111 sentences...
Processed 27008/529111 se

,review_id,sentence_num,sentiment,sentence
0,1548,2,2.0,I just going to enjoy while it lasts.
1,1,2,2.0,This exerciser certainly checked all the boxes...
2,1,3,2.0,The seat position is extremely comfortable and...
3,1,4,2.0,Provides a good range of resistance from super...
4,1,5,2.0,And the resistance bands and the multiple posi...


In [ ]:
# Example: Save locally as CSV
output_path = "/content/sentiment_results.csv"
df.to_csv(output_path, index=False)
print(f"Saved results to {output_path}")

Saved results to /content/sentiment_results.csv
